# Advanced Feature Store: Datasets, Online Serving & Governance

**Session 2 -- Deep Dive: Feature Engineering, Feature Stores & Experiment Tracking**  
**Notebook 2 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Point-in-Time Correctness** | Generate training datasets that prevent data leakage |
| **Immutable Datasets** | Create versioned, reproducible snapshots of feature data |
| **Dataset Lineage** | Trace the full provenance chain from raw table to dataset |
| **Online vs. Offline** | Understand the two-store architecture and when to use each |
| **Dynamic Refresh** | Configure online store freshness with `target_lag` |
| **Cross-Team Sharing** | Use Snowflake RBAC to share features without copying data |
| **Training-Serving Skew** | Eliminate it by using a single Feature View definition everywhere |

### Prerequisites

Run **01_feature_engineering_fundamentals.ipynb** first. It creates the `PATIENT` entity and
`PATIENT_FEATURES` Feature View that this notebook builds on.

---

## The Data Leakage Problem

When building training datasets for time-series or event-driven models, a common mistake is
joining features computed *after* the prediction point. This is **data leakage** -- your model
trains on future information it would not have at inference time.

```
  Timeline:
  --------|-----------|-----------|--------->
       t=1         t=2         t=3
       Event A     Event B     Event C

  If predicting at t=2, only features from t<=2 should be used.
  Using Event C's features at t=2 = DATA LEAKAGE
```

Snowflake's `generate_dataset()` with `spine_timestamp_col` performs a **point-in-time AS-OF join**
that automatically prevents this. For each row in your spine, it retrieves only features whose
timestamp is <= the spine timestamp.

## 1 | Environment Setup

Reconnect to the session and Feature Store created in notebook 1.

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import logging

from utils import get_config
from utils import get_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
config = get_config("config.yaml")

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse

# == Create Snowflake Session ==
session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(f"Context: {DB}.{SCHEMA} | Warehouse: {WH}")

### 1.1 Get the Feature View

In [ ]:
from snowflake.ml.feature_store import CreationMode, FeatureStore

fs = FeatureStore(
    session=session,
    database=DB,
    name=SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

registered_fv = fs.get_feature_view("PATIENT_FEATURES", "v1")
print(f"Feature View: {registered_fv.name}/{registered_fv.version} -- {registered_fv.status}")

## 2 | Point-in-Time Correct, Immutable Datasets

### The Spine Concept

A **spine** is the set of (entity_key, timestamp) pairs you want to generate features for.
Think of it as the "who + when" of your prediction events.

```
  Spine DataFrame:            Feature View (Dynamic Table):
  +------------+------------+  +------------+------------+----------+
  | PATIENT_ID | TIMESTAMP  |  | PATIENT_ID | TIMESTAMP  | FEATURES |
  +------------+------------+  +------------+------------+----------+
  | P001       | 2025-01-15 |  | P001       | 2025-01-10 | ...      |
  | P002       | 2025-02-20 |  | P001       | 2025-01-20 | ...      | <-- NOT used for P001
  +------------+------------+  | P002       | 2025-02-18 | ...      |
                               +------------+------------+----------+
```
  For P001 at 2025-01-15, only features with timestamp <= 2025-01-15 are joined.
  The row from 2025-01-20 is excluded -- preventing data leakage.

### Immutability & Versioning

The `generate_dataset()` call below does **two things at once**: it performs a point-in-time
correct feature join *and* persists the result as an **immutable, versioned Dataset** (Parquet
files under the hood). Once a version is created it can never be modified.

```
  Dataset: PATIENT_RISK_TRAINING_DS
  +-- Version: v_20260429_120000  (Parquet files, immutable)
  +-- Version: v_20260429_130000  (Parquet files, immutable)
```

Why this matters:

- **Reproducibility** -- re-train a model on the exact same data months later
- **Auditability** -- prove to regulators which data trained which model
- **Comparison** -- compare model performance across different data versions

Each version is accessible via `snow://dataset/` URIs and integrates with PyTorch,
TensorFlow, and Snowpark ML DataConnectors.

In [ ]:
from snowflake.snowpark import functions as F

raw_table = config.full_raw_table

spine_df = (
    session.table(raw_table)
    .select("PATIENT_ID", "TIMESTAMP")
    .with_column("_SPLIT", F.uniform(F.lit(0), F.lit(1), F.random()))
)

train_spine = spine_df.filter(F.col("_SPLIT") < 0.8).drop("_SPLIT")
test_spine = spine_df.filter(F.col("_SPLIT") >= 0.8).drop("_SPLIT")

print(f"Train spine rows: ~{train_spine.count():,}")
print(f"Test spine rows:  ~{test_spine.count():,}")
train_spine.show(5)

### Creating the Training Dataset

`generate_dataset()` performs the point-in-time join and persists the result as a versioned,
immutable Dataset with automatic lineage -- all in one call.

In [ ]:
from datetime import datetime, timezone

DATASET_NAME = "PATIENT_RISK_TRAINING_DS"
DATASET_VERSION = datetime.now(timezone.utc).strftime("v_%Y%m%d_%H%M%S")

training_dataset = fs.generate_dataset(
    spine_df=train_spine,
    features=[registered_fv],
    spine_timestamp_col="TIMESTAMP",
    name=DATASET_NAME,
    version=DATASET_VERSION,
    desc=f"Point-in-time training snapshot for Patient Risk model -- {DATASET_VERSION}",
)

print(f"Dataset created: {DATASET_NAME}/{DATASET_VERSION}")
print(f"Rows: {training_dataset.read.to_snowpark_dataframe().count():,}")

### Why `generate_dataset` for Training but `retrieve_feature_values` for Test?

We use **`generate_dataset`** for training data because it creates an **immutable, versioned snapshot** registered in Snowflake — giving us reproducibility, lineage tracking, and the ability to reload the exact same data later.

For test data, we use **`retrieve_feature_values`** instead and write it to a regular table. The test set doesn't need dataset-level versioning or lineage — it's an ephemeral evaluation artifact. Using `retrieve_feature_values` still gives us **point-in-time correct** feature joins against the spine, but without the overhead of registering a formal dataset object.

In [ ]:
test_table = f"{DB}.{SCHEMA}.TEST_FEATURES"

test_df = fs.retrieve_feature_values(
    spine_df=test_spine,
    features=[registered_fv],
    spine_timestamp_col="TIMESTAMP",
)
test_df.write.mode("overwrite").save_as_table(test_table)

print(f"Test table written: {test_table} ({session.table(test_table).count():,} rows)")

In [ ]:
print("Training dataset sample (notice: point-in-time correct features):")
training_dataset.read.to_snowpark_dataframe().select(
    "PATIENT_ID",
    "TIMESTAMP",
    "HEART_RATE",
    "SYSTOLIC_BP",
    "SHOCK_INDEX",
    "PULSE_PRESSURE",
    "VITAL_SIGNS_SEVERITY",
    "RISK_LEVEL",
).show(5)

### Creating a Second Version

To demonstrate versioning, we create another dataset version from a fresh random split.
Both versions coexist under the same dataset name -- each is immutable and independently loadable.

In [ ]:
DATASET_VERSION_2 = datetime.now(timezone.utc).strftime("v2_%Y%m%d_%H%M%S")

training_dataset_v2 = fs.generate_dataset(
    spine_df=train_spine,
    features=[registered_fv],
    spine_timestamp_col="TIMESTAMP",
    name=DATASET_NAME,
    version=DATASET_VERSION_2,
    desc=f"Second training snapshot (same spine, new version) -- {DATASET_VERSION_2}",
)

print(f"Version 2 created: {DATASET_NAME}/{DATASET_VERSION_2}")
print(f"Rows: {training_dataset_v2.read.to_snowpark_dataframe().count():,}")

In [ ]:
print("All dataset versions:")
session.sql(f"SHOW VERSIONS IN DATASET {DB}.{SCHEMA}.{DATASET_NAME}").show()

### Load a versioned dataset
This dataset is immutable. The exact same rows will be returned even if RAW_PATIENT_DATA has changed since creation.

In [ ]:
from snowflake.ml.dataset import load_dataset

reloaded = load_dataset(session, f"{DB}.{SCHEMA}.{DATASET_NAME}", version=DATASET_VERSION)
print(f"Loaded version: {reloaded.selected_version.name}")
print(f"Rows: {reloaded.read.to_snowpark_dataframe().count():,}")

## 3 | Dataset Lineage

Snowflake automatically tracks the full provenance chain:

```
  RAW_PATIENT_DATA (Table)
       |
       v
  PATIENT_FEATURES/v1 (Feature View / Dynamic Table)
       |
       v
  PATIENT_RISK_TRAINING_DS (Dataset, multiple versions)
       |
       v
  PATIENT_RISK_MODEL (Model -- notebook 4)
```

This lineage is captured automatically when you use `generate_dataset()`. No manual annotation required.

### Key Benefit

No manual annotation is needed. The lineage chain from raw tables through Feature Views to
Datasets to Models is captured end-to-end automatically through Snowflake's native ML APIs.

In [ ]:
print("Upstream lineage for the training dataset:")
print("(What data sources fed into this dataset?)")
print()

upstream = training_dataset.lineage(direction="upstream")
for node in upstream:
    print(f"  {node._lineage_node_domain:15s} : {node._lineage_node_name}")

## 4 | Online vs. Offline Feature Store

Snowflake's Feature Store has a **two-store architecture**:

```
                         Feature View Definition
                         (single source of truth)
                                  |
                    +-------------+-------------+
                    |                           |
              Offline Store                Online Store
          (Dynamic Table)            (Hybrid Table)
          Seconds latency            Milliseconds latency
          Full history               Latest values only
                    |                           |
            Training, batch              Real-time serving,
            inference, analytics         production apps
```

### Key Differences

| Aspect | Offline Store | Online Store |
|--------|---------------|-------------|
| **Backing** | Dynamic Table | Hybrid Table |
| **Latency** | Seconds | ~30ms p50, ~50ms p95 |
| **Data** | Full history | Latest values per entity key |
| **Access pattern** | Full scan or spine join | Key-based point lookup |
| **Use case** | Training, batch inference | Real-time serving |
| **Cost** | Standard DT compute | Additional Hybrid Table storage |

### No Training-Serving Skew

Both stores are populated from the **same Feature View definition**. This is the key insight:
features are computed once and served in two modes. There is no separate serving pipeline
that could drift out of sync.

In [ ]:
from snowflake.ml.feature_store import OnlineConfig

online_config = OnlineConfig(enable=True, target_lag="1 minutes")

updated_fv = fs.update_feature_view(
    name="PATIENT_FEATURES",
    version="v1",
    online_config=online_config,
)

print(f"Online serving enabled: {updated_fv.online}")
print(f"Target lag: 1 minute")
print()
print("The online store (Hybrid Table) will be provisioned and synced.")
print("Initial sync may take 30-60 seconds...")

In [ ]:
import time

from snowflake.ml.feature_store.feature_view import StoreType

# print("Waiting 30s for online store initial sync...")
# time.sleep(30)

sample_keys = session.table(raw_table).select("PATIENT_ID").limit(3).collect()
lookup_keys = [[row[0]] for row in sample_keys]

print(f"Looking up {len(lookup_keys)} patients from the ONLINE store:")
print(f"Keys: {lookup_keys}")
print()

try:
    online_result = fs.read_feature_view(
        "PATIENT_FEATURES",
        "v1",
        keys=lookup_keys,
        feature_names=["SHOCK_INDEX", "PULSE_PRESSURE", "VITAL_SIGNS_SEVERITY", "RISK_LEVEL"],
        store_type=StoreType.ONLINE,
    )
    online_result.show()
except Exception as e:
    print(f"Online store may still be provisioning. Error: {e}")
    print("Retry in 30 seconds or check the Hybrid Table status.")

## 5 | Dynamic Refresh of Online Store

The `target_lag` parameter controls how fresh the online store data is:

| Use Case | target_lag | Refresh Frequency | Cost Impact |
|----------|-----------|-------------------|-------------|
| Fraud detection | `15s` | Very frequent | Higher |
| Recommendations | `1m` | Frequent | Moderate |
| Marketing scores | `5m` | Less frequent | Lower |

The online store automatically syncs from the offline Dynamic Table.
When new data arrives in `RAW_PATIENT_DATA`:

```
  RAW_PATIENT_DATA  --[1 min refresh]--> Dynamic Table (offline)
                                              |
                                     --[target_lag]--> Hybrid Table (online)
```

Both refreshes are fully managed -- no orchestration code needed.

In [ ]:
print("Comparing offline vs online feature values for the same keys:")
print()

print("--- OFFLINE (Dynamic Table) ---")
offline_result = fs.read_feature_view(
    "PATIENT_FEATURES",
    "v1",
    keys=lookup_keys,
    feature_names=["SHOCK_INDEX", "PULSE_PRESSURE", "VITAL_SIGNS_SEVERITY"],
    store_type=StoreType.OFFLINE,
)
offline_result.show()

print("--- ONLINE (Hybrid Table) ---")
try:
    online_result2 = fs.read_feature_view(
        "PATIENT_FEATURES",
        "v1",
        keys=lookup_keys,
        feature_names=["SHOCK_INDEX", "PULSE_PRESSURE", "VITAL_SIGNS_SEVERITY"],
        store_type=StoreType.ONLINE,
    )
    online_result2.show()
    print("Values should match -- both stores serve the same Feature View definition.")
except Exception as e:
    print(f"Online store not yet ready: {e}")

## 6 | Eliminating Training-Serving Skew

Training-serving skew occurs when the features used during training differ from those
used during inference. This is one of the most insidious bugs in production ML.

### How Snowflake Prevents It

```
                   PATIENT_FEATURES Feature View
                   (SINGLE definition of all feature logic)
                              |
            +-----------------+-----------------+
            |                                   |
     generate_dataset()              retrieve_feature_values()
     (training time)                 (inference time)
            |                                   |
     Point-in-time correct            Latest feature values
     immutable snapshot               from same Dynamic Table
            |                                   |
     Train model                     Serve predictions
```

Both training and inference read from the **same Feature View**. The feature engineering
logic is defined once and materialized once. There is no separate "serving pipeline" that
could compute features differently.

### Common Skew Sources (and how Feature Store prevents them)

| Skew Source | Without Feature Store | With Feature Store |
|-------------|----------------------|--------------------|
| Different code paths | Training script vs. serving API | Single FeatureView definition |
| Different data sources | Training CSV vs. production DB | Same Dynamic Table |
| Schema drift | Columns renamed in production | Feature View enforces schema |
| Stale features | Batch job failed silently | Dynamic Table refresh is monitored |

## 7 | Cleanup: Disable Online Serving

Online serving incurs additional cost via the Hybrid Table. For this demo, we will
disable it to avoid unnecessary charges. In production, you would leave it enabled
for any Feature View on the real-time inference path.

In [ ]:
disable_config = OnlineConfig(enable=False)

fs.update_feature_view(
    name="PATIENT_FEATURES",
    version="v1",
    online_config=disable_config,
)

print("Online serving disabled for PATIENT_FEATURES/v1.")
print("Offline store (Dynamic Table) remains active.")

## 8 | Summary

In this notebook we:

1. **Generated** a point-in-time correct training dataset using `spine_timestamp_col`
2. **Created** multiple immutable dataset versions for reproducibility
3. **Traced** the lineage chain from raw table through feature view to dataset
4. **Enabled** online serving and compared offline vs. online reads
5. **Demonstrated** how target_lag controls online store freshness
6. **Showed** how RBAC sharing lets teams discover and reuse features
7. **Explained** how the single Feature View definition eliminates training-serving skew

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Point-in-time joins** | `spine_timestamp_col` prevents data leakage automatically |
| **Immutable datasets** | Once created, a version never changes -- perfect reproducibility |
| **Automatic lineage** | No manual annotation needed; captured by `generate_dataset()` |
| **Two-store architecture** | Same definition serves both batch (offline) and real-time (online) |
| **RBAC sharing** | Standard Snowflake grants -- no data copying or exporting |
| **Zero skew** | Single Feature View definition for training and serving |

### Variables Passed to Next Notebooks

The following were created and will be used in notebooks 3 and 4:

- **Dataset**: `PATIENT_RISK_TRAINING_DS` (version stored in `DATASET_VERSION`)
- **Test table**: `TEST_FEATURES`
- **Feature View**: `PATIENT_FEATURES/v1`

---

**Next -->** [03_experiment_tracking.ipynb](03_experiment_tracking.ipynb) -- Baseline runs, experiment variations, and comparing results